# Stochastic Calculus, Step by Step · Lesson 7: Information and measure

Companion notebook for the lesson. Every number in the lesson's text, tables and figures is produced by running this notebook and read from `build/outputs.json`.

> **Part 2's teaching tree.** A share at S₀ = 100 that, every four months, goes up 10% (heads, H) or down 10% (tails, T); three steps make the course's one year. The real-world chance of heads is p = 0.6. The risk-free rate is the course's r = 3% (continuously compounded), so cash grows by R = e^0.01 per step, and the strike is K = 105. Made up for teaching; every random number is simulated with a fixed seed.
>
> Educational material only, not financial advice.

1. Setup · 2. The coin-toss model · 3. Information as a partition · 4. Which events are in a σ-algebra · 5. Measurability: what you can compute from the information · 6. Probability on the tree · 7. Infinitely many tosses · 8. Exercises · 9. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

S0, K, r, sigma, T = 100.0, 105.0, 0.03, 0.20, 1.0     # the course's numbers
N = lambda x: 0.5 * (1 + math.erf(x / math.sqrt(2)))   # standard normal distribution function
out = {"meta": {
    "series": "Stochastic Calculus, Step by Step",
    "lesson": 7,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}, "inputs": {"S0": S0, "K": K, "r": r, "sigma": sigma, "T": T, "r_pct": r * 100, "sigma_pct": sigma * 100}}

In [ ]:
import itertools

S0_, u, d, p = 100.0, 1.1, 0.9, 0.6          # the teaching tree: +10% (H) or -10% (T) per step, P(H) = 0.6
steps, dt_ = 3, 1 / 3                         # three steps of four months
R = math.exp(r * dt_)                         # cash growth per step
worlds = ["".join(w) for w in itertools.product("HT", repeat=steps)]   # HHH, HHT, ..., TTT


def price(w, n):
    """The share price after the first n tosses of world w."""
    return S0_ * u ** w[:n].count("H") * d ** w[:n].count("T")


def prob(w, q=p):
    """The probability of world w (or of the first tosses of one) when heads has probability q."""
    return q ** w.count("H") * (1 - q) ** w.count("T")


out["tree"] = {"S0": S0_, "u": u, "d": d, "p": p, "steps": steps, "dt": dt_, "R": R, "K": K}
pd.DataFrame({"world": worlds, "S1": [price(w, 1) for w in worlds], "S2": [price(w, 2) for w in worlds],
              "S3": [price(w, 3) for w in worlds], "P": [prob(w) for w in worlds]}).round(4)

## 2. The coin-toss model

A **world** (an outcome $\omega$) is a sequence of three tosses, such as HTH. There are $2^3 = 8$ of them, and $\Omega$ is the set of all eight. The share price after $n$ tosses, $S_n(\omega)$, is $100 \times 1.1^{\#H} \times 0.9^{\#T}$ counted over the first $n$ tosses. Tosses are independent with $P(H) = 0.6$, so a world's probability is $0.6^{\#H}\,0.4^{\#T}$.

In [ ]:
rows = []
for w in worlds:
    path = [price(w, n) for n in range(steps + 1)]
    rows.append({"world": w, "S1": path[1], "S2": path[2], "S3": path[3], "P": prob(w), "M3": max(path),
                 "above": path[3] > K, "heads": w.count("H")})
out["worlds"] = rows
out["model"] = {"n_worlds": len(worlds), "total_p": sum(r_["P"] for r_ in rows),
                "p_hhh": prob("HHH"), "p_hth": prob("HTH"), "p_ttt": prob("TTT"),
                "s3_values": sorted({round(r_["S3"], 6) for r_ in rows}, reverse=True)}
assert abs(out["model"]["total_p"] - 1) < 1e-12
out["model"]["nodes"] = [{"t": n, "k": k_, "price": S0_ * u**k_ * d ** (n - k_)} for n in range(steps + 1) for k_ in range(n + 1)]
pd.DataFrame(rows).round(4)

## 3. Information as a partition

After $n$ tosses you know the first $n$ letters of the world, and nothing else. So the eight worlds fall into **blocks**: worlds that agree on their first $n$ tosses, which you cannot yet tell apart. The blocks get finer as tosses arrive: 1 block, then 2, 4, 8.

An event is **known at time $n$** (you can say whether it happened) exactly when it is a union of time-$n$ blocks. The collection of all such events is $\mathcal F_n$, the **σ-algebra** of time-$n$ information. A partition into $b$ blocks gives $2^b$ events (each block in or out).

In [ ]:
def blocks(n):
    # Group the worlds by their first n tosses.
    groups = {}
    for w in worlds:
        groups.setdefault(w[:n], []).append(w)
    return list(groups.values())


def events(bl):
    # Every union of blocks: one event for each choice of blocks in or out.
    evs = []
    for pick in itertools.product([0, 1], repeat=len(bl)):
        evs.append(frozenset(w for b_, k_ in zip(bl, pick) if k_ for w in b_))
    return evs


info = {"blocks": {n: blocks(n) for n in range(steps + 1)}}
info["n_blocks"] = [len(info["blocks"][n]) for n in range(steps + 1)]
info["n_events"] = [len(set(events(info["blocks"][n]))) for n in range(steps + 1)]
assert info["n_events"] == [2 ** b_ for b_ in info["n_blocks"]]
info["F1_events"] = [sorted(e) for e in sorted(set(events(info["blocks"][1])), key=lambda e: (len(e), sorted(e)))]
info["rows"] = [{"n": n, "blocks": info["n_blocks"][n], "events": info["n_events"][n]} for n in range(steps + 1)]
out["info"] = info
for n in range(steps + 1):
    print(f"after {n} tosses: {info['n_blocks'][n]} blocks, {info['n_events'][n]} events:", info["blocks"][n])

## 4. Which events are in a σ-algebra

A **σ-algebra** $\mathcal F$ on $\Omega$ is a collection of events that (1) contains $\Omega$, (2) contains the complement of each of its events, and (3) contains the union of any (countable) list of its events. These are exactly the rules "information" must follow: if you can tell whether $A$ happened, you can tell whether it didn't; if you can tell for $A$ and for $B$, you can tell for "$A$ or $B$".

Check the rules by brute force for each $\mathcal F_n$, and check that the information only grows: $\mathcal F_0 \subseteq \mathcal F_1 \subseteq \mathcal F_2 \subseteq \mathcal F_3$ (a **filtration**).

In [ ]:
Omega = frozenset(worlds)


def is_sigma_algebra(F):
    F = set(F)
    return (Omega in F and all(Omega - A in F for A in F)
            and all(A | B in F for A in F for B in F))


F = {n: set(events(info["blocks"][n])) for n in range(steps + 1)}
checks = {n: is_sigma_algebra(F[n]) for n in F}
nested = all(F[n] <= F[n + 1] for n in range(steps))
# A collection that is NOT a σ-algebra: "first toss H" without its complement
bad = {frozenset(), Omega, frozenset(w for w in worlds if w[0] == "H")}
out["sigma"] = {"all_ok": all(checks.values()), "nested": nested, "bad_ok": is_sigma_algebra(bad)}
# The information in the price S2 alone: HT and TH give the same price, so they stay together
by_s2 = {}
for w in worlds:
    by_s2.setdefault(round(price(w, 2), 6), []).append(w)
sig_s2 = set(events(list(by_s2.values())))
out["sigma"].update({"s2_blocks": [{"S2": k_, "worlds": v} for k_, v in by_s2.items()], "s2_n_events": len(sig_s2),
                     "s2_inside_F2": sig_s2 <= F[2], "F2_inside_s2": F[2] <= sig_s2})
out["sigma"]

## 5. Measurability: what you can compute from the information

A random variable $X$ is **$\mathcal F_n$-measurable** when its value is known at time $n$: it takes one value on each time-$n$ block. A process $(X_0, X_1, \dots)$ with each $X_n$ known at time $n$ is **adapted**. The table says, for several random variables, at which times they are known.

In [ ]:
def known(X, n):
    # X is F_n-measurable: constant on every time-n block.
    return all(len({round(X(w), 9) for w in b_}) == 1 for b_ in info["blocks"][n])


candidates = {
    "S1": lambda w: price(w, 1), "S2": lambda w: price(w, 2), "S3": lambda w: price(w, 3),
    "max(S0, S1, S2)": lambda w: max(price(w, n) for n in range(3)),
    "1{S3 > 105}": lambda w: float(price(w, 3) > K),
    "number of heads in the first two": lambda w: w[:2].count("H"),
}
meas = [{"name": k_, **{f"n{n}": known(X, n) for n in range(steps + 1)},
         "first": next(n for n in range(steps + 1) if known(X, n))} for k_, X in candidates.items()]
out["measurable"] = meas
# When is the event {S3 > 105} settled, world by world?
settled = []
for w in worlds:
    blk = lambda n: next(b_ for b_ in info["blocks"][n] if w in b_)
    settled.append({"world": w, "above": price(w, 3) > K,
                    "known_at": next(n for n in range(steps + 1) if len({price(v, 3) > K for v in blk(n)}) == 1)})
out["settled"] = settled
pd.DataFrame(meas)

## 6. Probability on the tree

A **probability measure** gives each event in $\mathcal F$ a number with three rules: $P(A) \ge 0$; $P(\Omega) = 1$; and for disjoint events, $P(A_1 \cup A_2 \cup \cdots) = P(A_1) + P(A_2) + \cdots$. On the tree it is enough to give each world a weight, and add.

In [ ]:
P = lambda A: sum(prob(w) for w in A)
above = frozenset(w for w in worlds if price(w, 3) > K)
first_h = frozenset(w for w in worlds if w[0] == "H")
prob_ = {"above": P(above), "above_formula": 3 * p**2 * (1 - p) + p**3,
         "first_h": P(first_h), "both": P(above & first_h), "either": P(above | first_h),
         "at_least_one_h": 1 - (1 - p) ** 3, "at_least_one_h_sum": P(frozenset(w for w in worlds if "H" in w)),
         "above_fair": sum(0.5 ** 3 for w in above)}
assert abs(prob_["either"] - (prob_["above"] + prob_["first_h"] - prob_["both"])) < 1e-12
assert abs(prob_["above"] - prob_["above_formula"]) < 1e-12
# simulate 100,000 years of three tosses
rng = np.random.default_rng(seed)
tosses = rng.random((100_000, steps)) < p
s3 = S0_ * u ** tosses.sum(1) * d ** (steps - tosses.sum(1))
prob_["sim_above"] = float((s3 > K).mean())
prob_["sim_first_h"] = float(tosses[:, 0].mean())
out["prob"] = prob_
out["events"] = {"above": sorted(above), "first_h": sorted(first_h),
                 "s2_99": sorted(w for w in worlds if abs(price(w, 2) - 99) < 1e-9)}
prob_

## 7. Infinitely many tosses

With infinitely many tosses there are uncountably many worlds and each one has probability 0, so we cannot simply add up world weights; and it turns out that we cannot give a sensible probability to every subset of $\Omega$ at once. That is why probability is defined on a σ-algebra of events. Two facts the simulation shows: the share of heads settles at $p$ (Lesson 6), and "heads every time" is possible but has probability $0.6^n \to 0$.

In [ ]:
long = rng.random(100_000) < p
running = np.cumsum(long) / np.arange(1, long.size + 1)
out["infinite"] = {"share_heads": {str(n): float(running[n - 1]) for n in (10, 100, 1000, 10_000, 100_000)},
                   "all_heads": {str(n): p ** n for n in (1, 10, 50, 100)}}
idx = np.unique(np.round(np.geomspace(1, 2000, 300)).astype(int))   # the first 2,000 tosses, for a chart
out["infinite"]["fig"] = {"n": idx.tolist(), "share": running[idx - 1].tolist(), "p": [p] * idx.size,
                          "ticks": [{"x": x, "label": f"{x:,}"} for x in (0, 500, 1000, 1500, 2000)]}
out["infinite"]

## 8. Exercises

Try each one before looking anything up. Full solutions are in the companion volume, *Part 2: Solutions*.

**Exercise 1.** List the four events in $\mathcal F_1$ and check the three rules of a σ-algebra for them.

**Exercise 2.** Show that a partition of $\Omega$ into $b$ blocks generates exactly $2^b$ events. How many events are in $\mathcal F_n$ for the $n$-toss tree? How many digits does the number of events in $\mathcal F_{10}$ have?

**Exercise 3.** Let $A = \{S_3 > 105\}$. Show that $A \notin \mathcal F_2$, and find, world by world, the first time at which you know whether $A$ happened.

**Exercise 4.** Find the blocks of $\sigma(S_2)$. Show that $\sigma(S_2) \subsetneq \mathcal F_2$ (contained in, but not equal to) by naming an event in $\mathcal F_2$ that is not in $\sigma(S_2)$. What do you lose by watching only the price?

**Exercise 5.** From the three rules of a probability measure, prove that $P(\emptyset) = 0$, $P(A^c) = 1 - P(A)$ and $P(A \cup B) = P(A) + P(B) - P(A \cap B)$. Then compute $P(S_3 > 105)$ and $P(S_3 > 105 \text{ or the first toss is H})$ on the teaching tree.

**Exercise 6.** Let $\tau$ be the time of the first tail ($\tau = 4$ if there is none in three tosses). Show that $\{\tau \le n\} \in \mathcal F_n$ for each $n$, but that $\{\tau = 1\} \cup \{\tau = 3\}$ is not in $\mathcal F_2$. Compute $P(\tau = n)$ for $n = 1, 2, 3, 4$.

**Exercise 7.** (Code) For the four-toss tree, build $\mathcal F_2$ as all unions of blocks and check by brute force that it is a σ-algebra with $2^4 = 16$ events.

**Exercise 8.** (Code) Simulate 200,000 years of three tosses with $p = 0.6$. Estimate the probability of each time-2 block and of $\{S_3 > 105\}$, and compare with the exact values.

## 9. Export

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")